In [ ]:
# Setup

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

!wget -nc https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

device = 'cuda' if torch.cuda.is_available() else 'cpu'

if device == 'cuda' and torch.cuda.is_bf16_supported():
    dtype = torch.bfloat16
elif device == 'cuda':
    dtype = torch.float16
else:
    dtype = torch.float32

print(f"{device=}\n")
print(f"{dtype=}\n")

# config
lr = 4e-3
train_iter = 10000
batch_size = 64
dim = 64
n_heads = 4
head_dims = dim // n_heads
n_layers = 4
mlp_dim = 1024
block_size = 256
eval_batches = 10
show_plots = True

with open("input.txt", "r") as f:
    text = f.read()

# char tokenizer
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c:i for i,c in enumerate(chars)}
itos = {i:c for c,i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(f"{vocab_size=}\n")
print(f"{dim=}\n")
print(f"{n_layers=}\n")

# split data: 80% train, 10% val, 10% test
tokens = torch.tensor(encode(text), device=device)
n = len(tokens)
train_tokens = tokens[:int(0.8 * n)]
val_tokens = tokens[int(0.8 * n):int(0.9 * n)]
test_tokens = tokens[int(0.9 * n):]

print(f"Train = {len(train_tokens)}")
print(f"Val = {len(val_tokens)}")
print(f"Test = {len(test_tokens)}\n")

# sliding windows per split
train_seqs = train_tokens.unfold(0, block_size + 1, 1)
val_seqs = val_tokens.unfold(0, block_size + 1, 1)
test_seqs = test_tokens.unfold(0, block_size + 1, 1)

emb = torch.randn(vocab_size, dim, device=device, dtype=dtype) * 0.02
pos = torch.randn(block_size, dim, device=device, dtype=dtype) * 0.02

# one set of weights per layer
init_scale = 0.1 / (2 ** 0.5)
wq = [torch.randn(dim, dim, device=device, dtype=dtype) * init_scale for _ in range(n_layers)]
wk = [torch.randn(dim, dim, device=device, dtype=dtype) * init_scale for _ in range(n_layers)]
wv = [torch.randn(dim, dim, device=device, dtype=dtype) * init_scale for _ in range(n_layers)]
wo = [torch.randn(dim, dim, device=device, dtype=dtype) * init_scale for _ in range(n_layers)]
w1 = [torch.randn(dim, mlp_dim, device=device, dtype=dtype) * init_scale for _ in range(n_layers)]
w2 = [torch.randn(mlp_dim, dim, device=device, dtype=dtype) * init_scale for _ in range(n_layers)]

params = [emb, pos] + wq + wk + wv + wo + w1 + w2
for p in params:
    p.requires_grad = True

def rmsnorm(x, eps=1e-5):
    return x / ((x ** 2).mean(dim=-1, keepdim=True).sqrt() + eps)

def forward(x):
    B, T = x.shape
    x = emb[x] + pos[:T]
    mask = torch.tril(torch.ones(T, T, device=device, dtype=torch.bool))

    for layer in range(n_layers):
        nx = rmsnorm(x)

        q, k, v = nx @ wq[layer], nx @ wk[layer], nx @ wv[layer]
        q = q.view(B,T,n_heads,head_dims).transpose(1,2)
        k = k.view(B,T,n_heads,head_dims).transpose(1,2)
        v = v.view(B,T,n_heads,head_dims).transpose(1,2)

        scores = q @ k.transpose(-2, -1) / (head_dims ** 0.5)
        scores = scores.masked_fill(~mask, float('-inf'))
        attn_out = (F.softmax(scores, dim=-1) @ v).transpose(1,2).contiguous().view(B, T, dim)
        x = x + attn_out @ wo[layer]

        nx = rmsnorm(x)
        x = x + (nx @ w1[layer]).relu() @ w2[layer]

    x = rmsnorm(x)
    x = x @ emb.T
    return x

@torch.no_grad()
def eval_loss(seqs, n_batches=eval_batches):
    total = 0.0
    for _ in range(n_batches):
        idx = torch.randint(0, seqs.shape[0], (batch_size,))
        x, y = seqs[idx, :-1], seqs[idx, 1:]
        total += F.cross_entropy(forward(x).view(-1, vocab_size), y.view(-1)).item()
    return total / n_batches

forward_raw = forward
if hasattr(torch, 'compile'):
    forward = torch.compile(forward)
    print("Using torch.compile")

opt = torch.optim.Adam(params, lr=lr, fused=(device == 'cuda'))

In [ ]:
# Save / Load Weights

def save_weights(path="shakesp_minimal_transformer_weights.pt"):
    torch.save({
        'emb': emb, 'pos': pos,
        'wq': wq, 'wk': wk, 'wv': wv, 'wo': wo,
        'w1': w1, 'w2': w2,
    }, path)
    print(f"Saved to {path}")

def load_weights(path="shakesp_minimal_transformer_weights.pt"):
    global emb, pos, wq, wk, wv, wo, w1, w2, params
    ckpt = torch.load(path, map_location=device, weights_only=True)
    emb, pos = ckpt['emb'], ckpt['pos']
    wq, wk, wv, wo = ckpt['wq'], ckpt['wk'], ckpt['wv'], ckpt['wo']
    w1, w2 = ckpt['w1'], ckpt['w2']
    params = [emb, pos] + wq + wk + wv + wo + w1 + w2
    for p in params:
        p.requires_grad = True
    print(f"Loaded from {path}")

# load_weights()

In [ ]:
# Train Loop

plot_every = 1000
train_losses, val_losses = [], []

def plot_embeddings(step, loss_val):
    E = emb.detach().float().cpu()
    E = E - E.mean(dim=0)
    U, S, V = torch.svd(E)
    coords = E @ V[:, :2]
    plt.figure(figsize=(8, 8))
    plt.scatter(coords[:, 0].numpy(), coords[:, 1].numpy(), s=20)
    for i, c in itos.items():
        label = repr(c) if c in (' ', '\n', '\t') else c
        plt.annotate(label, (coords[i, 0].item(), coords[i, 1].item()), fontsize=11)
    plt.title(f"Step {step} | Loss {loss_val:.2f}")
    plt.grid(True, alpha=0.3)
    plt.show()

eval_every = 10
for i in range(train_iter):
    idx = torch.randint(0, train_seqs.shape[0], (batch_size,))
    x, y = train_seqs[idx, :-1], train_seqs[idx, 1:]

    logits = forward(x)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))

    opt.zero_grad()
    loss.backward()
    opt.step()

    if i % eval_every == 0:
        train_l = loss.item()
        val_l = eval_loss(val_seqs)
        print(f"{i}: train={train_l:.2f}, val={val_l:.2f}")
        if show_plots:
            train_losses.append((i, train_l))
            val_losses.append((i, val_l))

    if show_plots and i % plot_every == 0:
        plot_embeddings(i, loss.item())

test_loss = eval_loss(test_seqs, n_batches=50)
print(f"Test loss: {test_loss:.2f}")

if show_plots:
    steps_t, losses_t = zip(*train_losses)
    steps_v, losses_v = zip(*val_losses)
    plt.figure(figsize=(10, 4))
    plt.plot(steps_t, losses_t, label='train')
    plt.plot(steps_v, losses_v, label='val')
    plt.axhline(y=test_loss, color='r', linestyle='--', label=f'test={test_loss:.2f}')
    plt.xlabel('step'); plt.ylabel('loss'); plt.legend()
    plt.title('Train / Val / Test Loss')
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
save_weights()

In [ ]:
# Generate

ctx = "Firs"
tokens = encode(ctx)

for i in range(200):
    x = torch.tensor([tokens[-block_size:]], device=device)
    logits = forward_raw(x)
    probs = F.softmax(logits[0, -1]/0.8, dim=-1)
    next_token = torch.multinomial(probs, 1).item()
    tokens.append(next_token)
print(decode(tokens))